# 27_cited_joint: text and reference lists of cited works outside the pulled set, in one pull (full run, S4.0)

Thin driver: logic lives in `citecurv.data.cited`. Runtime: **CPU** (no GPU needed), Python 3.12 or newer.

Why: `10_data` stores a record only for the works that match the slice filter. The works they cite are ids
without title, abstract, date or reference list. The TGN needs their text (PROGRESS.md, S1.7b) and the curvature
part needs their reference lists (S2.2). `25_cited_text` and `26_cited_refs` fetched the two separately on the
pilot, one call per 100 ids each. This notebook fetches both in the same request
(`pull_works_by_ids(what="text_and_references")`), which halves the calls of the full run.

Reads from `MyDrive/citecurv/<RUN>/`: `events.parquet`, `works.parquet`.
Writes there: `cited_joint.parquet` (work_id, title, abstract, pub_date, n_refs, ref_ids), `cited_joint_stats.json`,
`cited_joint_record.json`, and while the pull is unfinished the folder `cited_joint_parts/` (one file per 5,000 ids).
Needs the Colab secret `OPENALEX_API_KEY`.

Cost: no money. One OpenAlex list call per 100 ids of the free key's daily allowance of 10,000 calls (reset at
00:00 UTC). The full run needs several days: when the allowance is spent the pull cell stops with the reset time,
and a rerun of all cells after the reset continues with the parts that are missing.

The last step merges every part in memory. For a few million works use a High-RAM runtime for the run that
finishes the pull (the fetching days themselves need little memory).


In [ ]:
# ---- Python version gate -----------------------------------------------------------------------------
import platform
import sys

print("python  ", sys.version)
if sys.version_info < (3, 12):
    raise RuntimeError(f"This notebook needs Python 3.12 or newer, found {platform.python_version()}.")


In [ ]:
# ---- install citecurv with the data-stage pins (configs/pins.md) ------------------------------------------
import subprocess

REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
REPO_TAG = "main"   # must contain what="text_and_references" (c3ca510 or later); set a release tag for the final run
subprocess.run(["rm", "-rf", "/content/repo"], check=True)   # a rerun clones again
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_TAG, REPO, "/content/repo"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
%pip install -q /content/repo "pyalex==0.21" pyarrow requests pyyaml

import importlib.metadata as md

VERSIONS = {"python": sys.version.split()[0], "commit": COMMIT}
for pkg in ("citecurv", "pyalex", "numpy", "pandas", "pyarrow"):
    VERSIONS[pkg] = md.version(pkg)
print(VERSIONS)


In [ ]:
# ---- Drive, API key (Colab Secrets), settings ------------------------------------------------------------
import json
import os
import shutil
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["OPENALEX_API_KEY"] = userdata.get("OPENALEX_API_KEY")   # never print or save it

from citecurv.config import load_config

cfg = load_config("/content/repo/configs/default.yaml")
RUN = "full"                                    # same value as in 10_data
RUN_DIR = Path("/content/drive/MyDrive") / "citecurv" / RUN
EVENTS = RUN_DIR / "events.parquet"
WORKS = RUN_DIR / "works.parquet"
CITED_JOINT = RUN_DIR / "cited_joint.parquet"
assert EVENTS.exists() and WORKS.exists(), f"run 10_data with RUN = {RUN!r} first"
DRIVE_FREE_GB = round(shutil.disk_usage("/content/drive/MyDrive").free / 2**30, 1)
print(RUN_DIR, "| Drive free:", DRIVE_FREE_GB, "GB")


In [ ]:
# ---- which cited works have no record ----------------------------------------------------------------------
import pandas as pd

from citecurv.data.cited import cited_only_ids
from citecurv.data.events import read_events

events = read_events(EVENTS)
works = pd.read_parquet(WORKS, columns=["work_id"])
ids = cited_only_ids(events, works)
print(f"{len(events):,} events | {events['dst_id'].nunique():,} cited works | {len(ids):,} of them without a record")


In [ ]:
# ---- OpenAlex allowance left today, and what this pull needs ---------------------------------------------------
import math

import requests

r = requests.get("https://api.openalex.org/rate-limit", params={"api_key": os.environ["OPENALEX_API_KEY"]}, timeout=30)
print(r.status_code, (r.json() if r.ok else {}).get("rate_limit"))
CALLS_NEEDED = math.ceil(len(ids) / 100)
print("calls needed:", CALLS_NEEDED, "| days of a 10,000-call allowance:", math.ceil(CALLS_NEEDED / 10_000))


In [ ]:
# ---- trial: one request must return text and reference lists together (1 call) ---------------------------------
from citecurv.data.cited import JOINT_COLUMNS, pull_works_by_ids

ID_FILTER = "openalex"   # found by the live probe of 25_cited_text on 2026-11-12
TRIAL_IDS = list(works["work_id"].head(3))   # pulled works: known to exist
TRIAL_OUT = Path("/content/trial_joint.parquet")
for old in (TRIAL_OUT, TRIAL_OUT.with_name("trial_joint_stats.json")):
    old.unlink(missing_ok=True)   # a rerun asks again
trial_stats = pull_works_by_ids(TRIAL_IDS, TRIAL_OUT, config=cfg, id_filter=ID_FILTER, what="text_and_references")
trial = pd.read_parquet(TRIAL_OUT)
print(trial_stats)
print(trial.assign(title=trial["title"].str.slice(0, 40), abstract_chars=trial["abstract"].fillna("").str.len(),
                   refs_listed=trial["ref_ids"].map(len))[["work_id", "title", "abstract_chars", "pub_date", "n_refs", "refs_listed"]])
assert list(trial.columns) == JOINT_COLUMNS and trial_stats["n_found"] == len(TRIAL_IDS), "the joint request did not return the sample"
assert trial["title"].notna().any() and trial["pub_date"].notna().all(), "text fields are missing from the joint answer"


In [ ]:
# ---- fetch text and reference lists by id (resumable: finished parts are kept in cited_joint_parts/) -------------
import time

from citecurv.data.openalex import BudgetExhausted

t0 = time.perf_counter()
try:
    pull_stats = pull_works_by_ids(ids, CITED_JOINT, config=cfg, id_filter=ID_FILTER, what="text_and_references", log=print)
except BudgetExhausted as e:
    print(e)   # reset time (UTC); finished parts are saved, rerun this notebook after the reset
    parts = sorted((RUN_DIR / "cited_joint_parts").glob("part_*.parquet"))
    print(f"{len(parts)} of {math.ceil(len(ids) / 5000)} parts are saved; {round((time.perf_counter() - t0) / 60, 1)} minutes this session")
    raise
pull_stats["minutes"] = round((time.perf_counter() - t0) / 60, 2)
print(pull_stats)


In [ ]:
# ---- what came back, and how much of the event table it covers -------------------------------------------------
import pyarrow.parquet as pq

joint = pd.read_parquet(CITED_JOINT, columns=["work_id", "n_refs"])
n_refs = joint.set_index("work_id")["n_refs"]
outside = ~events["dst_id"].isin(works["work_id"])
got = events["dst_id"].map(n_refs)
record = {
    "run": RUN, "versions": VERSIONS, "pull": pull_stats, "trial": trial_stats,
    "n_events": int(len(events)), "n_events_cited_outside": int(outside.sum()),
    "n_events_cited_outside_with_record": int((outside & got.notna()).sum()),
    "n_events_cited_outside_count_zero": int((outside & got.eq(0)).sum()),
    "n_refs_quantiles": {str(q): float(v) for q, v in joint["n_refs"].quantile([0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).items()},
    "rows": int(pq.ParquetFile(CITED_JOINT).metadata.num_rows),
    "file_mb": round(CITED_JOINT.stat().st_size / 2**20, 1),
    "drive_free_gb_at_start": DRIVE_FREE_GB,
}
(RUN_DIR / "cited_joint_record.json").write_text(json.dumps(record, indent=1))
print(json.dumps(record, indent=1))
